# ResNet50 (8 通道) 盐碱地分类训练

8 波段 Sentinel-1/2 patch 的 patch 级二分类。ImageNet 预训练 + 微调, 5-fold CV。

**用法**: 选 GPU runtime (T4) -> `Runtime > Run all` -> 全自动跑完出结果。

数据来自 Google Drive `saline_export_batch_v1/` (60 tif + labels_v1.csv + features.csv)。

## 1. 环境检查 + 挂载 Drive

In [ ]:
# 检查 GPU
import torch
assert torch.cuda.is_available(), "需要 GPU runtime, 请切换到 T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

# 挂载 Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone 项目 + 装依赖

In [ ]:
import os
os.chdir('/content')

# Clone GitHub repo
!git clone https://github.com/kejiaz22/saline-rs.git
os.chdir('/content/saline-rs')

# 装 Colab 额外依赖
!pip install -q -r requirements-colab.txt

## 3. 链接 Drive 数据到项目结构

In [ ]:
import os
import shutil

# Drive 路径
DRIVE_DATA = '/content/drive/MyDrive/saline_export_batch_v1'

# 目标路径 (项目内)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/labels', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# 软链 60 tif 到 data/raw/ (用 symlink 不复制, 省时间)
tif_count = 0
for f in os.listdir(DRIVE_DATA):
    if f.endswith('.tif'):
        src = os.path.join(DRIVE_DATA, f)
        dst = os.path.join('data/raw', f)
        if not os.path.exists(dst):
            os.symlink(src, dst)
        tif_count += 1
print(f"\u2705 Linked {tif_count} tif files")

# 复制 CSV (小文件, 直接复制更稳)
shutil.copy(os.path.join(DRIVE_DATA, 'labels_v1.csv'), 'data/labels/labels_v1.csv')
shutil.copy(os.path.join(DRIVE_DATA, 'features.csv'), 'data/processed/features.csv')
print("\u2705 Copied labels_v1.csv and features.csv")

# 验证
assert tif_count == 60, f"Expected 60 tifs, got {tif_count}"
assert os.path.exists('data/labels/labels_v1.csv')
assert os.path.exists('data/processed/features.csv')
print("\u2705 All data ready")

## 4. 训练 (5-fold)

In [ ]:
# 把项目根加到 PYTHONPATH 让 import 能找到 config
import sys
sys.path.insert(0, '/content/saline-rs')

# 跑训练
!python -m src.deeplearning.train

## 5. 保存结果回 Drive

In [ ]:
import shutil
import os

# 把 results 整个目录拷回 Drive (方便本地后续 git pull 时同步)
DRIVE_RESULTS = '/content/drive/MyDrive/saline-rs-results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

if os.path.exists('results/resnet50_results.json'):
    shutil.copy('results/resnet50_results.json', DRIVE_RESULTS)
    print(f"\u2705 Saved results to {DRIVE_RESULTS}")
else:
    print("\u26a0\ufe0f No results file found")

# 列出 results 目录确认
!ls -la results/

## 6. 结果摘要

In [ ]:
import json

with open('results/resnet50_results.json') as f:
    results = json.load(f)

print(json.dumps(results, indent=2))